In [ ]:
# Import Block
!pip install qiskit
!pip install qiskit_aer
import random
import math
import numpy as np
import qiskit as qc
from qiskit import QuantumCircuit, QuantumRegister
from qiskit.quantum_info import Operator, Statevector

In [ ]:
# Internal Function Definitions
def DimensionGiver(n):
  N = n
  Possible_Pairs = []
  for i in range(2,N):
    if (N % i == 0):
      if (i <= (N / i)):
        Possible_Pairs.append([i, int(N / i)])
      else:     #This Stuff is so that we define width as the smaller value, and height as larger value to consistantly be able to compute
        Possible_Pairs.append([int(N / i), i])
  Dimension = random.choice(Possible_Pairs)
  return Dimension


def GiveHammingWeight(x):
  BitX = bin(x)[2:]
  return BitX.count("1")

def GiveParity(x):
  return GiveHammingWeight(x) % 2

def KeyChoice(dimension):
  x, y = dimension
  target_parity = GiveParity(x) ^ GiveParity(y)

  repeat = True
  while repeat:
    key_x = random.randrange(2, x+1, 2)  #We do this now, because out lord and saviour, PDF, has told us to use even superposition :D
    key_y = random.randint(1, y)
    if (GiveParity(key_x) ^ GiveParity(key_y)) == target_parity:
      repeat = False

  return [key_x, key_y]


def GiveQubitCount(dimension):
  x, y = dimension
  QubitCount = [int(np.ceil(np.log2(x))),int(np.ceil(np.log2(y)))]
  return QubitCount

In [ ]:
# Automated Space Preparation
def Initiate():
  NumberChoices = [4, 6, 9, 10, 14, 15, 21, 22, 25, 26, 33, 34, 35, 38, 39, 46, 49, 51, 55, 57, 58, 62, 65, 69, 74, 77, 82, 85, 86, 87, 91, 93, 94, 95, 106, 111, 115, 118, 119, 121, 122, 123, 129, 133, 134, 141, 142, 143, 145, 146, 155, 158, 159, 161, 166, 169, 177, 178, 183, 185, 187, 194, 201, 202, 203, 205, 206, 209, 213, 214, 215, 217, 218, 219, 221, 226, 235, 237, 247, 249, 253, 254]
  N = random.choice(NumberChoices)
  Dimension = DimensionGiver(N)
  print(f"Dimension: {Dimension}")
  print(f"Key : {KeyChoice(Dimension)}")
  register_A = QuantumRegister(GiveQubitCount(Dimension)[0], name='Width')
  register_B = QuantumRegister(GiveQubitCount(Dimension)[1], name='Height')
  ancilla = QuantumRegister(3, name='Ancilla')
  circuit = QuantumCircuit(register_A, register_B)
  print(f"Given N: {N}")
  return circuit, N


#Here, build unitary matrix
def BuildUnitary(Gamma, N):
  n = int(np.ceil(np.log2(N)))
  M = 2**n

  U = np.zeros((M, M))

  for y in range(M):
    if (y < N):
      output = (Gamma * y) % N
    else:
      output = y
    U[output][y] = 1
  return Operator(U)


#The function which runs iterations of unitary until the invariant is found
def RunIter(n,U,N,Y):

  Y_int = int(Y,2)
  print(f"Y = {Y_int}")

  qc = QuantumCircuit(n)
  for i in range(len(Y)):
          if Y[i] == '1':
              qc.x(len(Y) - 1 - i)

  for i in range(N):

    qc.unitary(U, range(n))
    Y_new = Statevector.from_instruction(qc)
    if ((np.argmax(np.abs(Y_new.data)) == Y_int) & (i > 1)):
      return (i+1)


#The master Call function which, given a total dimension N, gives the dimensions of the space
def ProvideDimensions(N):
    n = int(np.ceil(np.log2(N)))

    valid_Gamma = []
    for i in range(2, N):
        if math.gcd(i, N) == 1:
            valid_Gamma.append(i)

    Gamma = random.choice(valid_Gamma)
    U = BuildUnitary(Gamma, N)

    Y_val = random.randint(1, N)
    Y = format(Y_val, 'b')

    r = RunIter(n, U, N, Y)
    if r is None:
        return None

    x = pow(int(Gamma), r // 2, int(N))
    if x == 1:
        return None

    Final1 = math.gcd(N, int(x + 1))
    Final2 = math.gcd(N, int(x - 1))
    for f in (Final1, Final2):
        if f > 1 and N % f == 0:
            other = N // f
            if other > 1:
                return (f, other)

    return None



In [ ]:
Space, N = Initiate()

dims = None
while dims is None:
    dims = ProvideDimensions(N)
Dim1, Dim2 = dims

if(Dim1 <= Dim2):
  Width = Dim1
  Height = Dim2
else:
  Wdith = Dim2
  Height = Dim1

print(f"Width is: {Width}, and Height is: {Height}")


Dimension: [3, 7]
Key : [2, 6]
Given N: 21
Y = 3
Width is: 3, and Height is: 7


Phase 1 Complete, moving on to phase 2

In [ ]:
#Defining the function which creates even superposition
from qiskit.circuit.library import MCPGate
def CreateConstrainedEvenSuperposition(Space, N, width):
  n = int(np.ceil(np.log2(N)))
  targets = [q for q in range(n) if q != (width - 1)]
  Space.h(targets)
  return Space


EvenSpace = CreateConstrainedEvenSuperposition(Space, N, Width)
EvenSpace.draw('text')

#Remember to change how key is chosen to fit the HW(x) XOR HW(y) = HW(W|H)
#Remember to check only the parity
#Remember to use minimum ancilla
#Remember that the purpose of phase 2 is only too reduce states by a factor of 2, phase 3 finds keystate

def ApplyHammingWeightResonanceOracle(Space, width, n, target_parity):
    anc_px, anc_py, target = n, n + 1, n + 2
    x_qubits = list(range(0, width))
    y_qubits = list(range(width, n))

    for q in x_qubits:
        Space.cx(q, anc_px)
    for q in y_qubits:
        Space.cx(q, anc_py)

    #combine both parities into anc_px: anc_px now = HW(x) + HW(y)
    Space.cx(anc_py, anc_px)

    # if target==1, flip anc_px once so "1" always means "condition holds"
    if target_parity == 1:
        Space.x(anc_px)

    Space.cx(anc_px, target)

    if target_parity == 1:
        Space.x(anc_px)
    Space.cx(anc_py, anc_px)
    for q in reversed(y_qubits):
        Space.cx(q, anc_py)
    for q in reversed(x_qubits):
        Space.cx(q, anc_px)

    return Space
def build_pi3_diffuser(num_search_qubits):

    qc = QuantumCircuit(num_search_qubits, name='D_π/3')
    
    qc.h(range(num_search_qubits))
    qc.x(range(num_search_qubits))
    
    mcp = MCPGate(np.pi / 3, num_ctrl_qubits=num_search_qubits - 1)
    qc.append(mcp, list(range(num_search_qubits)))
    
    qc.x(range(num_search_qubits))
    qc.h(range(num_search_qubits))
    
    return qc.to_instruction()


key = KeyChoice((4,4))
print(f"Key: {key}")
print(GiveHammingWeight(key[0]) % 2, GiveHammingWeight(key[1]) % 2)
search_qubits_count = Width + Height

EvenSpace.x(search_qubits_count + 2)
EvenSpace.h(search_qubits_count + 2)

OracleSpace = ApplyHammingWeightResonanceOracle(EvenSpace, Width, search_qubits_count, 1)

diffuser = build_pi3_diffuser(search_qubits_count)
OracleSpace.append(diffuser, list(range(search_qubits_count)))
import random
for _ in range(30):
  dim = (random.randint(2,20), random.randint(2,20))
  print(dim, KeyChoice(dim))

Key: [4, 2]
1 1
(16, 6) [6, 2]
(2, 9) [2, 9]
(4, 2) [2, 2]
(8, 4) [4, 1]
(16, 9) [14, 9]
(11, 15) [8, 9]
(9, 20) [2, 2]
(5, 19) [2, 9]
(7, 2) [2, 1]
(8, 19) [4, 19]
(8, 4) [6, 3]
(16, 10) [14, 5]
(20, 11) [14, 9]
(19, 15) [8, 5]
(17, 16) [16, 5]
(12, 3) [6, 3]
(17, 14) [4, 6]
(9, 2) [6, 2]
(18, 20) [6, 3]
(8, 7) [2, 2]
(13, 4) [4, 2]
(11, 12) [6, 7]
(14, 2) [4, 1]
(16, 19) [10, 12]
(3, 7) [2, 3]
(3, 13) [2, 9]
(17, 18) [12, 12]
(4, 11) [2, 2]
(9, 13) [4, 12]
(2, 2) [2, 2]
